# A — Khảo sát BioDCASE 2026

**Mục tiêu:** kiểm tra dữ liệu tải về, phân bố loài–domain, chất lượng cấu trúc audio và khả năng dùng window ngắn; đối chiếu với HumBugDB trước khi chọn loài.

Notebook đọc WAV trực tiếp từ Downloads, không thay đổi dữ liệu gốc và không huấn luyện model. Chạy **Run All** bằng môi trường `.venv` của project. Lần đầu quét toàn bộ Development có thể lâu; các lần sau dùng cache nếu kích thước và thời điểm sửa file không đổi.

**Quy ước:** clip là một file WAV; window là đoạn cắt dự kiến; cả hai đều không đồng nghĩa với cá thể muỗi hoặc nguồn ghi độc lập. `domain` biểu diễn điều kiện nguồn thu, không mặc định là một microphone.

Nguồn: [BioDCASE Task 5](https://biodcase.github.io/challenge2026/task5), `metadata/split_summary.json`, các danh sách ID được tải kèm và inventory HumBugDB trong project. Ánh xạ species ID theo trang Task 5; notebook kiểm tra lại số lượng bằng metadata local.

Evaluation chỉ kiểm tra số file và sự hiện diện của nhãn; không phân tích nội dung, không dùng để chọn thiết lập. Test trong Development chỉ được kiểm tra tính toàn vẹn và mô tả phân bố, không dùng để lựa chọn mô hình.

## 1. Cấu hình và thư mục kết quả
Các đường dẫn đọc từ `config.json`. Thiết lập chính là window 300 ms, hop 150 ms; số window là khả năng cắt theo độ dài, chưa loại nhiễu hoặc khoảng không có tiếng muỗi.

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor
import os, re, json, hashlib, platform
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, Audio

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config.json').exists())
CONFIG = json.loads((ROOT / 'config.json').read_text(encoding='utf-8-sig'))
BIO = CONFIG['biodcase']
DEV = Path(BIO['development_dir'])
EVAL = Path(BIO['evaluation_dir'])
OUT = ROOT / BIO['results_dir']
TABLES, FIGURES = OUT / 'tables', OUT / 'figures'
for folder in [TABLES, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)
assert (DEV / 'raw_audio').is_dir() and (DEV / 'metadata').is_dir(), DEV
assert EVAL.is_dir(), EVAL
WINDOW_SECONDS, HOP_SECONDS = 0.300, 0.150
REPORTED_DEVELOPMENT_SECONDS = 218388.40  # Trang Task 5; đối chiếu với WAV local
FORCE_SCAN = False
WORKERS = 4
SPECIES = {1:'Aedes aegypti', 2:'Aedes albopictus', 3:'Culex quinquefasciatus',
           4:'Anopheles gambiae', 5:'Anopheles arabiensis', 6:'Anopheles dirus',
           7:'Culex pipiens', 8:'Anopheles minimus', 9:'Anopheles stephensi'}
TARGETS = ['Anopheles dirus','Anopheles minimus','Aedes aegypti','Aedes albopictus']
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi':120, 'axes.spines.top':False, 'axes.spines.right':False})
pd.set_option('display.max_columns', 18)

def save_table(frame, name):
    frame.to_csv(TABLES / f'{name}.csv', index=False, encoding='utf-8-sig')
    return frame

def finish_plot(name):
    plt.tight_layout()
    plt.savefig(FIGURES / f'{name}.png', dpi=160, bbox_inches='tight')
    plt.show()
    plt.close()

print('Development:', DEV)
print('Evaluation:', EVAL)
print('Kết quả:', OUT)

## 2. Kiểm tra file và các split chính thức
Đối chiếu WAV với ID, tìm ID trùng, split giao nhau, file dư và tên không hợp lệ. `TrainVal` là hợp của training và validation, **không phải tập thứ tư để cộng thêm vào tổng**. Nếu kiểm tra cấu trúc thất bại, dừng trước khi phân tích tiếp.

In [ ]:
official = json.loads((DEV / 'metadata/split_summary.json').read_text(encoding='utf-8'))
list_files = {'train':'Training_ids.txt','validation':'Validation_ids.txt',
              'test':'Test_ids.txt','trainval':'TrainVal_ids.txt'}
ids = {k:[Path(x.strip()).stem for x in (DEV/'metadata'/v).read_text().splitlines() if x.strip()]
       for k,v in list_files.items()}
id_sets = {k:set(v) for k,v in ids.items()}
entries = []
with os.scandir(DEV/'raw_audio') as scan:
    for e in scan:
        if e.is_file() and e.name.lower().endswith('.wav') and not e.name.startswith('._'):
            st = e.stat()
            entries.append({'clip_id':Path(e.name).stem, 'path':e.path, 'bytes':st.st_size,
                            'mtime_ns':st.st_mtime_ns})
files = pd.DataFrame(entries).sort_values('clip_id').reset_index(drop=True)
actual = set(files.clip_id)
checks = []
for split, values in ids.items():
    checks.append({'check':f'{split}: count vs metadata', 'issues':abs(len(values)-official['counts'][split])})
    checks.append({'check':f'{split}: duplicate IDs', 'issues':len(values)-len(set(values))})
    checks.append({'check':f'{split}: missing WAV', 'issues':len(id_sets[split]-actual)})
for a,b in [('train','validation'),('train','test'),('validation','test')]:
    checks.append({'check':f'{a} / {b}: shared IDs', 'issues':len(id_sets[a]&id_sets[b])})
checks += [
    {'check':'TrainVal != train union validation', 'issues':len(id_sets['trainval']^(id_sets['train']|id_sets['validation']))},
    {'check':'WAV outside official splits', 'issues':len(actual-(id_sets['trainval']|id_sets['test']))},
    {'check':'WAV count vs metadata', 'issues':abs(len(files)-official['counts']['all_total'])}]
parsed = files.clip_id.str.extract(r'^S_(?P<species_id>\d+)_D_(?P<domain_id>\d+)_(?P<clip_index>\d+)$')
checks.append({'check':'Invalid file naming', 'issues':int(parsed.isna().any(axis=1).sum())})
check_table = save_table(pd.DataFrame(checks), 'integrity_checks')
display(check_table)
assert check_table.issues.sum() == 0, 'Sửa lỗi cấu trúc trước khi phân tích.'
files = pd.concat([files, parsed.astype(int)], axis=1)
files['species'] = files.species_id.map(SPECIES)
assert files.species.notna().all()
assert files.domain_id.between(1,5).all(), 'Domain ngoài ánh xạ công bố'
files['domain'] = 'D' + files.domain_id.astype(str)
assignment = {clip:s for s in ['train','validation','test'] for clip in ids[s]}
files['split'] = files.clip_id.map(assignment)
files['held_out_domain_for_species'] = files.species.map(official['unseen_domain_by_species'])
files['is_held_out_species_domain'] = files.domain.eq(files.held_out_domain_for_species)
for row in official['species_summary']:
    assert int(files.species.eq(row['species']).sum()) == row['species_total'], row['species']
with os.scandir(EVAL) as scan:
    eval_names = [e.name for e in scan if e.is_file() and not e.name.startswith('.')]
eval_wavs = [x for x in eval_names if x.lower().endswith('.wav')]
eval_other = [x for x in eval_names if not x.lower().endswith('.wav')]
split_counts = save_table(files.groupby('split',sort=False).size().rename('clips').reset_index(), 'split_counts')
display(split_counts)
print(f'Development: {len(files):,} WAV; Evaluation: {len(eval_wavs):,} WAV.')
print('File ngoài WAV trong thư mục Evaluation:', eval_other)
print('Chưa có nhãn Evaluation trong thư mục audio: không tính metric hoặc dùng cho tuning.')

## 3. Phân bố loài, domain và split
Biểu đồ tổng thể giúp nhận diện mất cân bằng, không dùng số clip để suy ra số cá thể. Heatmap training trả lời: model được nhìn thấy mỗi loài ở những domain nào? Ô bằng 0 là thiếu dữ liệu, không thể khắc phục chỉ bằng tăng trọng số loss.

Bảng đếm test được giữ để mô tả protocol đã công bố; mọi khuyến nghị lựa chọn loài bên dưới dùng training.

In [ ]:
species_split = pd.crosstab(files.species, files.split).reindex(columns=['train','validation','test'],fill_value=0)
save_table(species_split.reset_index(), 'species_by_split')
domain_split = pd.crosstab(files.domain, files.split).reindex(columns=['train','validation','test'],fill_value=0)
save_table(domain_split.reset_index(), 'domain_by_split')
save_table(files.groupby(['species','domain','split']).size().rename('clips').reset_index(), 'species_domain_split')
fig, ax = plt.subplots(1,2,figsize=(14,5))
species_split.sum(axis=1).sort_values().plot.barh(ax=ax[0],color='#247B91')
ax[0].set_xscale('log'); ax[0].set_xlabel('Số clip — thang log'); ax[0].set_ylabel('')
ax[0].set_title('Development: số clip theo loài')
domain_split.plot.bar(stacked=False,ax=ax[1],color=['#247B91','#E4A64A','#835C96'])
ax[1].set_yscale('log'); ax[1].set_ylabel('Số clip — thang log'); ax[1].set_xlabel('Domain')
ax[1].set_title('Development: phân bố domain và split'); ax[1].tick_params(axis='x',rotation=0)
finish_plot('01_species_domain_counts')
train_counts = pd.crosstab(files.loc[files.split.eq('train'),'species'], files.loc[files.split.eq('train'),'domain']).reindex(index=SPECIES.values(),columns=[f'D{i}' for i in range(1,6)],fill_value=0)
plt.figure(figsize=(10,6))
sns.heatmap(np.log10(train_counts+1),annot=train_counts,fmt='d',cmap='Blues',cbar_kws={'label':'log10(số clip + 1)'})
plt.title('Training: số clip theo loài × domain'); plt.xlabel('Domain'); plt.ylabel('')
finish_plot('02_training_species_domain')
display(species_split)

## 4. Quét header WAV và checksum có cache
Đọc sample rate, số kênh, số frame, subtype và thời lượng của **toàn bộ Development**. SHA-256 kiểm tra trùng byte trong BioDCASE và đối chiếu inventory HumBugDB.

Cache lưu trong `audio_audit.csv`, dùng đường dẫn, kích thước và thời điểm sửa file để kiểm tra thay đổi. Đặt `FORCE_SCAN=True` nếu nghi ngờ file đã thay đổi mà vẫn giữ timestamp. Đây là kiểm tra header và đọc byte; chưa phải kiểm tra toàn bộ chất lượng tín hiệu/SNR hoặc xác nhận mỗi clip đều có tiếng muỗi.

In [ ]:
CACHE = OUT / 'audio_audit.csv'
cache_columns = ['clip_id','path','bytes','mtime_ns','sample_rate','frames','channels','subtype','seconds','sha256','error']
old = pd.read_csv(CACHE,keep_default_na=False) if CACHE.exists() and not FORCE_SCAN else pd.DataFrame(columns=cache_columns)
lookup = old.drop_duplicates('clip_id',keep='last').set_index('clip_id').to_dict('index')
keep, pending = [], []
for row in files[['clip_id','path','bytes','mtime_ns']].to_dict('records'):
    prev = lookup.get(row['clip_id'])
    if prev and prev['path']==row['path'] and int(prev['bytes'])==row['bytes'] and int(prev['mtime_ns'])==row['mtime_ns'] and not prev['error']:
        keep.append({'clip_id':row['clip_id'],**prev})
    else:
        pending.append(row)

def inspect_wav(row):
    result = {**row,'sample_rate':0,'frames':0,'channels':0,'subtype':'','seconds':0.,'sha256':'','error':''}
    try:
        info = sf.info(row['path'])
        with open(row['path'],'rb') as handle:
            digest = hashlib.file_digest(handle,'sha256').hexdigest()
        result.update(sample_rate=info.samplerate,frames=info.frames,channels=info.channels,
                      subtype=info.subtype,seconds=info.frames/info.samplerate,sha256=digest)
        if info.frames<=0: result['error']='empty_audio'
    except Exception as exc:
        result['error']=f'{type(exc).__name__}: {exc}'
    return result

print(f'Cache hợp lệ: {len(keep):,}; cần quét: {len(pending):,}',flush=True)
pd.DataFrame(keep,columns=cache_columns).to_csv(CACHE,index=False)
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    for start in range(0,len(pending),5000):
        batch = list(pool.map(inspect_wav,pending[start:start+5000]))
        pd.DataFrame(batch,columns=cache_columns).to_csv(CACHE,mode='a',header=False,index=False)
        print(f'Đã quét {min(start+5000,len(pending)):,}/{len(pending):,}',flush=True)
audit = pd.read_csv(CACHE,keep_default_na=False)
assert len(audit)==len(files) and audit.clip_id.is_unique
for col in ['sample_rate','frames','channels','seconds']:
    audit[col]=pd.to_numeric(audit[col],errors='coerce')
data = files.merge(audit.drop(columns=['path','bytes','mtime_ns']),on='clip_id',validate='one_to_one')
valid = data.loc[data.error.eq('') & data.seconds.gt(0)].copy()
errors = save_table(data.loc[~data.index.isin(valid.index),['clip_id','error']], 'audio_errors')
print(f'Đọc được header + byte: {len(valid):,}/{len(data):,}; lỗi: {len(errors):,}')
save_table(valid.groupby(['sample_rate','channels','subtype']).agg(clips=('clip_id','size'),hours=('seconds',lambda s:s.sum()/3600)).reset_index(), 'audio_formats')
display(pd.read_csv(TABLES/'audio_formats.csv'))

## 5. Thời lượng và khả năng cắt window 300 ms
Với một clip có `N` frame, window `W` frame và hop `H` frame, số window đầy đủ là `max(0, 1 + floor((N-W)/H))`. Mỗi clip được cắt riêng, không nối hai clip. Window có overlap **không phải các mẫu độc lập**.

Đoạn ngắn hơn 300 ms được đếm riêng; notebook chưa tự pad, lặp tín hiệu hoặc loại bỏ. Kết quả dưới đây dùng training để lựa chọn thiết lập; chỉ báo tổng giờ cho Development để đối chiếu dữ liệu.

Boxplot bỏ các điểm ngoại lệ để dễ đọc; khi phần lớn clip có cùng độ dài, hộp có thể co thành một vạch. Cột window bằng 0 không hiển thị trên trục log. Tổng giờ đo từ WAV được đối chiếu với 218.388,40 giây trên trang Task 5; chênh lệch chưa được giải thích.

In [ ]:
for seconds, label in [(0.15,'150ms'),(0.3,'300ms'),(0.625,'625ms'),(1.0,'1000ms')]:
    w = np.rint(valid.sample_rate*seconds).astype('int64')
    h = np.maximum(1,np.rint(valid.sample_rate*seconds/2).astype('int64'))
    valid[f'windows_{label}'] = np.where(valid.frames>=w,1+(valid.frames-w)//h,0).astype('int64')
valid['short_300ms'] = valid.frames < np.rint(valid.sample_rate*WINDOW_SECONDS)
train = valid.loc[valid.split.eq('train')].copy()
training_stats = train.groupby('species').agg(clips=('clip_id','size'),seconds=('seconds','sum'),
    median_seconds=('seconds','median'),p10_seconds=('seconds',lambda s:s.quantile(.1)),
    p90_seconds=('seconds',lambda s:s.quantile(.9)),short_300ms_pct=('short_300ms',lambda s:100*s.mean()),
    full_windows_300ms=('windows_300ms','sum'),domains=('domain','nunique')).reindex(SPECIES.values())
training_stats['minutes']=training_stats.seconds/60
save_table(training_stats.reset_index(), 'training_duration_windows')
save_table(train.groupby(['species','domain']).agg(clips=('clip_id','size'),seconds=('seconds','sum'),windows_300ms=('windows_300ms','sum')).reset_index(), 'training_species_domain_duration')
display(training_stats.round(3))
print(f'Tổng thời lượng Development đọc được: {valid.seconds.sum()/3600:.3f} giờ')
print(f'Trang Task 5: {REPORTED_DEVELOPMENT_SECONDS/3600:.3f} giờ; local - công bố: {(valid.seconds.sum()-REPORTED_DEVELOPMENT_SECONDS)/60:.3f} phút. Chưa xác minh nguyên nhân.')
fig, ax=plt.subplots(1,2,figsize=(14,5))
sns.boxplot(data=train,x='seconds',y='species',order=list(SPECIES.values()),showfliers=False,ax=ax[0],color='#8CC4CD')
ax[0].set_xscale('log'); ax[0].axvline(.3,color='#B34747',linestyle='--',label='300 ms')
ax[0].set_xlabel('Thời lượng (giây, log)'); ax[0].set_ylabel(''); ax[0].legend(); ax[0].set_title('Training: phân bố thời lượng')
training_stats.short_300ms_pct.plot.barh(ax=ax[1],color='#D29249')
ax[1].invert_yaxis()
ax[1].bar_label(ax[1].containers[0],labels=[f'{v:.2f}%' for v in training_stats.short_300ms_pct],padding=4,fontsize=9)
ax[1].set_xlim(0,max(1,training_stats.short_300ms_pct.max()*1.3)); ax[1].set_xlabel('% clip ngắn hơn 300 ms'); ax[1].set_ylabel(''); ax[1].set_title('Training: clip cần quyết định cách xử lý')
finish_plot('03_training_duration')
window_comparison=train.groupby('species')[[f'windows_{x}' for x in ['150ms','300ms','625ms','1000ms']]].sum()
save_table(window_comparison.reset_index(),'training_window_comparison')
window_comparison.rename(columns=lambda s:s.replace('windows_','')).plot.bar(figsize=(12,5),logy=True)
plt.ylabel('Số window đầy đủ — log'); plt.xlabel(''); plt.xticks(rotation=25,ha='right')
plt.title('Training: số window dự kiến, overlap 50% — không phải số nguồn độc lập')
finish_plot('04_window_comparison')

## 6. Bốn loài mục tiêu và mức phụ thuộc domain
Nhóm mục tiêu là **đề xuất**, chưa chốt. So sánh cả số clip và thời lượng: nhiều clip ngắn không nhất thiết cung cấp nhiều âm thanh hơn. Domain lớn nhất chiếm tỷ lệ cao là dấu hiệu cần kiểm tra điều kiện thu; không chứng minh model đã học sai.

Không có nhãn nguồn gốc bản ghi, cá thể, giới tính hay microphone trong tên `S_*_D_*_*`. Không suy ra các trường này từ chỉ số clip hoặc domain.

In [ ]:
target_train = train.loc[train.species.isin(TARGETS)]
target_counts=pd.crosstab(target_train.species,target_train.domain).reindex(index=TARGETS,columns=[f'D{i}' for i in range(1,6)],fill_value=0)
target_seconds=target_train.pivot_table(index='species',columns='domain',values='seconds',aggfunc='sum',fill_value=0).reindex(index=TARGETS,columns=target_counts.columns,fill_value=0)
target_summary=training_stats.reindex(TARGETS).copy()
target_summary['largest_domain_clip_pct']=target_counts.max(axis=1)/target_counts.sum(axis=1)*100
target_summary['largest_domain_duration_pct']=target_seconds.max(axis=1)/target_seconds.sum(axis=1)*100
target_summary['independent_sources']='Chưa xác minh'
target_summary['gender']='Không có trong metadata kèm theo'
save_table(target_summary.reset_index(),'target_species_training')
display(target_summary.round(2))
fig,ax=plt.subplots(1,2,figsize=(13,4))
for matrix,axis,title in [(target_counts,ax[0],'Tỷ trọng clip theo domain'),(target_seconds,ax[1],'Tỷ trọng thời lượng theo domain')]:
    pct=matrix.div(matrix.sum(axis=1),axis=0)*100
    pct.plot.barh(stacked=True,ax=axis,colormap='tab10')
    axis.set_xlim(0,100); axis.set_xlabel('% trong từng loài — training'); axis.set_ylabel(''); axis.set_title(title)
    axis.legend(title='Domain',loc='upper center',bbox_to_anchor=(.5,-.17),ncol=5)
finish_plot('05_target_domain_shares')

## 7. Trùng lặp nội bộ và đối chiếu HumBugDB
**Hai mức kiểm tra:** (1) SHA-256 toàn file; (2) SHA-256 mẫu âm thanh giải mã float64 cho các file có cùng sample rate, số frame và số kênh với HumBugDB. Mức (2) có thể phát hiện nội dung giống nhau nhưng header khác, với cùng biểu diễn mẫu giải mã.

Không tự xóa hoặc gộp file. Những bản trùng qua split là cảnh báo cần xử lý trước huấn luyện. Kiểm tra này **chưa tìm đoạn con, bản resample, đổi gain, hoặc bản ghi bị cắt khác nhau**; không có match không chứng minh hai bộ độc lập. Giới tính, nguồn thu chỉ được chuyển nhãn sau khi xác minh phù hợp, không tự gán trong notebook.

In [ ]:
humbug = pd.read_json(ROOT/CONFIG['inventory'],lines=True)
humbug['id']=humbug.id.astype(str)
byte_duplicates=valid.loc[valid.sha256.duplicated(keep=False),['clip_id','species','domain','split','sha256']].sort_values('sha256')
save_table(byte_duplicates,'biodcase_byte_duplicates')
duplicate_groups=byte_duplicates.groupby('sha256').agg(clips=('clip_id','size'),splits=('split','nunique'),species=('species','nunique'))
cross_split_groups=int(duplicate_groups.splits.gt(1).sum())
duplicate_detail=byte_duplicates.groupby('sha256').agg(clips=('clip_id','size'),splits=('split',lambda s:', '.join(sorted(set(s)))),species=('species',lambda s:', '.join(sorted(set(s)))))
save_table(duplicate_detail.reset_index(),'duplicate_group_summary')
display(duplicate_detail.reset_index().drop(columns='sha256').head(20))
byte_matches=valid[['clip_id','species','domain','split','sha256']].merge(humbug[['id','relative_path','sha256']],on='sha256')
save_table(byte_matches,'humbug_byte_matches')
match_summary=byte_matches.groupby(['species','domain','split']).clip_id.nunique().rename('matched_biodcase_clips').reset_index()
save_table(match_summary,'humbug_match_summary')
display(match_summary)
print(f'Nhóm trùng byte nội bộ: {len(duplicate_groups):,}; qua nhiều split: {cross_split_groups:,}')
print(f'Cặp BioDCASE–HumBugDB trùng byte: {len(byte_matches):,}')

hkeys=set(zip(humbug.sample_rate,humbug.frames,humbug.channels))
pcm_candidates=valid.loc[[key in hkeys for key in zip(valid.sample_rate,valid.frames,valid.channels)]].copy()
bkeys=set(zip(pcm_candidates.sample_rate,pcm_candidates.frames,pcm_candidates.channels))
h_candidates=humbug.loc[[key in bkeys for key in zip(humbug.sample_rate,humbug.frames,humbug.channels)]].copy()

def pcm_digest(path):
    try:
        digest=hashlib.sha256()
        with sf.SoundFile(str(path)) as handle:
            digest.update(f'{handle.samplerate}|{handle.channels}|{len(handle)}|'.encode())
            for block in handle.blocks(blocksize=65536,dtype='float64',always_2d=True):
                digest.update(np.asarray(block,dtype='<f8').tobytes())
        return digest.hexdigest(),''
    except Exception as exc:
        return '',str(exc)

PCM_CACHE=OUT/'pcm_audit.csv'
previous=pd.read_csv(PCM_CACHE,keep_default_na=False) if PCM_CACHE.exists() and not FORCE_SCAN else pd.DataFrame(columns=['path','bytes','mtime_ns','pcm_sha256','error'])
pcm_lookup=previous.drop_duplicates('path',keep='last').set_index('path').to_dict('index')
paths=list(pcm_candidates.path)+[str(ROOT/p) for p in h_candidates.relative_path]
def inspect_pcm(path):
    st=Path(path).stat(); prev=pcm_lookup.get(path)
    if prev and int(prev['bytes'])==st.st_size and int(prev['mtime_ns'])==st.st_mtime_ns and not prev['error']:
        return {'path':path,**prev}
    digest,error=pcm_digest(path)
    return {'path':path,'bytes':st.st_size,'mtime_ns':st.st_mtime_ns,'pcm_sha256':digest,'error':error}
print(f'Đối chiếu mẫu giải mã: {len(pcm_candidates):,} clip BioDCASE và {len(h_candidates):,} clip HumBugDB',flush=True)
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    pcm_rows=list(pool.map(inspect_pcm,paths))
pcm_cache=pd.DataFrame(pcm_rows,columns=['path','bytes','mtime_ns','pcm_sha256','error'])
pcm_cache.to_csv(PCM_CACHE,index=False)
pcm_map=pcm_cache.loc[pcm_cache.error.eq('')].set_index('path').pcm_sha256.to_dict()
pcm_candidates['pcm_sha256']=pcm_candidates.path.map(pcm_map)
h_candidates['pcm_sha256']=[pcm_map.get(str(ROOT/p)) for p in h_candidates.relative_path]
pcm_matches=pcm_candidates.dropna(subset=['pcm_sha256'])[['clip_id','species','domain','split','pcm_sha256']].merge(
    h_candidates.dropna(subset=['pcm_sha256'])[['id','relative_path','pcm_sha256']],on='pcm_sha256')
save_table(pcm_matches,'humbug_pcm_matches')
print(f'Cặp trùng mẫu giải mã: {len(pcm_matches):,}; lỗi giải mã: {int(pcm_cache.error.ne("").sum()):,}')
display(pcm_matches.head(10))

# Kiểm tra nhóm trùng byte nhưng có nhiều nhãn loài.
conflict_hashes=duplicate_groups.index[duplicate_groups.species.gt(1)]
conflicts=byte_duplicates.loc[byte_duplicates.sha256.isin(conflict_hashes)].copy()
conflict_signals=[]
for digest, group in conflicts.groupby('sha256'):
    clip=group.iloc[0].clip_id
    samples,sr=sf.read(DEV/'raw_audio'/f'{clip}.wav',dtype='float64',always_2d=True)
    conflict_signals.append({'sha256':digest,'sample_rate':sr,'frames':len(samples),
        'peak':float(np.max(np.abs(samples))),'rms':float(np.sqrt(np.mean(samples*samples))),
        'all_zero':bool(np.all(samples==0))})
if conflict_signals:
    conflicts=conflicts.merge(pd.DataFrame(conflict_signals),on='sha256',validate='many_to_one')
save_table(conflicts,'conflicting_species_duplicates')
print(f'Nhóm trùng byte mang nhiều nhãn loài: {len(conflict_hashes)}; số clip: {len(conflicts)}')
display(conflicts.drop(columns='sha256'))
print('Không tự sửa nhãn. Peak/RMS chỉ phân biệt tín hiệu toàn zero, không xác định loài đúng.')


## 8. Nghe và xem một window mẫu cho từng loài mục tiêu
Chỉ chọn **training**: clip có đủ 300 ms và thời lượng gần trung vị trong nhóm đủ dài. Lấy window ở giữa clip để minh họa, không coi đây là đoạn có chất lượng tốt nhất. Không resample hay chuẩn hóa âm lượng giữa các clip khi phát.

Spectrogram dùng STFT 32 ms, hop 8 ms trên sample rate gốc; đây là minh họa, chưa chốt frontend cho model. Dải hiển thị đến 4 kHz hoặc Nyquist nếu thấp hơn. Nghe bốn mẫu không đủ để kết luận chất lượng toàn dataset.

Spectrogram dùng cùng thang màu −110 đến −20 dB/Hz (PSD theo biên độ số), không phải mức âm thanh dB SPL đo bằng microphone đã hiệu chuẩn.

In [ ]:
example_rows=[]
fig,axes=plt.subplots(len(TARGETS),2,figsize=(13,10))
for i,species in enumerate(TARGETS):
    candidates=train.loc[train.species.eq(species) & ~train.short_300ms].copy()
    if candidates.empty:
        for axis in axes[i]: axis.text(.5,.5,'Không có clip đủ dài',ha='center')
        continue
    candidates['distance']=(candidates.seconds-candidates.seconds.median()).abs()
    row=candidates.sort_values(['distance','clip_id']).iloc[0]
    n=int(round(WINDOW_SECONDS*row.sample_rate)); start=max(0,(int(row.frames)-n)//2)
    y,sr=sf.read(row.path,start=start,frames=n,dtype='float64',always_2d=True)
    mono=y.mean(axis=1)
    axes[i,0].plot(np.arange(len(mono))/sr,mono,linewidth=.7,color='#247B91')
    axes[i,0].set_title(f'{species} | {row.domain} | {row.clip_id}',fontsize=10)
    axes[i,0].set_xlabel('Giây'); axes[i,0].set_ylabel('Biên độ')
    nfft=max(32,int(round(.032*sr))); hop=max(1,int(round(.008*sr)))
    _,_,_,im=axes[i,1].specgram(mono,NFFT=nfft,Fs=sr,noverlap=nfft-hop,cmap='magma',vmin=-110,vmax=-20)
    axes[i,1].grid(False)
    fig.colorbar(im,ax=axes[i,1],label='PSD (dB/Hz)')
    axes[i,1].set_ylim(0,min(4000,sr/2)); axes[i,1].set_xlabel('Giây'); axes[i,1].set_ylabel('Hz')
    axes[i,1].set_title('STFT của window 300 ms',fontsize=10)
    display(Markdown(f'**{species}** — `{row.clip_id}`, {sr} Hz, window giữa clip.'))
    display(Audio(mono,rate=sr,normalize=False))
    example_rows.append({'species':species,'clip_id':row.clip_id,'split':'train','start_frame':start,'window_frames':n,'sample_rate':sr})
finish_plot('06_training_audio_examples')
save_table(pd.DataFrame(example_rows),'audio_examples')

## 9. Kết luận Step A và các quyết định còn mở
Phân biệt dữ kiện đo được với những thông tin chưa có. Notebook không chọn loài tự động và không tạo split mới. Trước huấn luyện cần xác minh nguồn bản ghi và các bản gần trùng, xử lý cảnh báo trùng qua split, quyết định chính sách giới tính và định nghĩa rõ custom benchmark hay tái lập split chính thức.

Nếu chỉ có domain mà thiếu nguồn độc lập, không thể gọi kết quả là đánh giá theo cá thể hoặc theo nguồn ghi. Baseline BioDCASE công bố đầu vào dài hơn 1,1 giây; nếu điều chỉnh cho 300 ms phải báo cáo là phiên bản điều chỉnh.

In [ ]:
summary={
    'run_time':datetime.now(ZoneInfo('Asia/Ho_Chi_Minh')).isoformat(),
    'development_dir':str(DEV),'evaluation_dir':str(EVAL),
    'development_wav':len(files),'evaluation_wav':len(eval_wavs),
    'official_split_counts':official['counts'],'structure_checks_passed':bool(check_table.issues.eq(0).all()),
    'audio_header_errors':len(errors),'development_hours':float(valid.seconds.sum()/3600),
    'reported_development_seconds':REPORTED_DEVELOPMENT_SECONDS,
    'duration_delta_seconds':float(valid.seconds.sum()-REPORTED_DEVELOPMENT_SECONDS),
    'duration_discrepancy_resolved':False,
    'byte_duplicate_groups':len(duplicate_groups),'byte_duplicate_cross_split_groups':cross_split_groups,
    'humbug_byte_matched_biodcase_clips':int(byte_matches.clip_id.nunique()),
    'humbug_pcm_matched_biodcase_clips':int(pcm_matches.clip_id.nunique()),
    'pcm_decode_errors':int(pcm_cache.error.ne('').sum()),
    'conflicting_species_duplicate_groups':len(conflict_hashes),
    'conflicting_species_duplicate_clips':len(conflicts),
    'window_seconds':WINDOW_SECONDS,'hop_seconds':HOP_SECONDS,
    'selected_species':[],'research_split_created':False,
    'provenance_verified':False,'near_duplicate_search_completed':False,
    'gender_available_in_biodcase_metadata':False,'evaluation_labels_found':False,
    'metadata_sha256':{name:hashlib.sha256((DEV/'metadata'/name).read_bytes()).hexdigest()
                       for name in [*list_files.values(),'split_summary.json']},
    'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'soundfile':sf.__version__}}
(OUT/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
conclusions=[
    f'Development có **{len(files):,} clip**, tổng **{valid.seconds.sum()/3600:.2f} giờ** đọc được; lỗi header/đọc byte: **{len(errors):,}**.',
    f'Có **{len(duplicate_groups):,} nhóm trùng byte**, trong đó **{cross_split_groups:,} nhóm qua nhiều split**. Cần xem bảng chi tiết trước huấn luyện.',
    f'Có **{len(conflict_hashes)} nhóm trùng byte mang nhiều nhãn loài**, gồm **{len(conflicts)} clip**. Cần xác minh nhãn gốc; không tự chọn nhãn theo đa số.',
    f'BioDCASE có **{byte_matches.clip_id.nunique():,} clip trùng byte** và **{pcm_matches.clip_id.nunique():,} clip trùng mẫu giải mã** với HumBugDB. Hai số này có thể chồng nhau, không cộng lại.',
    f'Tổng thời lượng local lệch **{(valid.seconds.sum()-REPORTED_DEVELOPMENT_SECONDS)/60:.2f} phút** so với trang Task 5; chưa xác minh nguyên nhân dù số lượng file khớp.',
    'Chưa xác minh nguồn gốc bản ghi/cá thể; chưa tìm gần trùng do cắt đoạn, resample hoặc đổi gain. Không coi phần chưa match là dữ liệu độc lập.',
    'Bốn loài vẫn là ứng viên. Số clip và window không thay thế cho số nguồn độc lập; nhãn giới tính cần bổ sung bằng chứng nguồn.',
    'Evaluation giữ riêng; chưa có nhãn local để tính metric. Không dùng để chọn frontend, window hoặc mô hình.']
for item in conclusions: display(Markdown('- '+item))
display(Markdown('**Tiếp theo:** thảo luận bảng bốn loài, giải quyết nguồn/trùng lặp và chính sách giới tính trước khi chốt dữ liệu cho phần B.'))